# Can AI Warn an Acquirer? Predicting goodwill-impairment failures in US acquisitions

AI for Finance capstone (Type A). This notebook reproduces the study from the repository.

- **Part A (2 minutes):** loads the saved results and shows every table and figure of the manuscript.
- **Part B (about 1-2 hours):** rebuilds everything from the raw public data (SEC, Yahoo, Kenneth French) and re-runs all models.

Licensed Bloomberg prices are not distributed. If you have them, upload `bloomberg_prices.xlsx` in step B3; otherwise the stock test runs on Yahoo prices only.

## Setup: get the code

In [ ]:
# Get the code: clone the GitHub repository (or set USE_ZIP = True to upload ma-failure-repo.zip instead)
REPO = "https://github.com/francoisdeffrennes/M-A-Failure.git"
USE_ZIP = False
if not USE_ZIP:
    !git clone -q $REPO project
    %cd project
else:
    from google.colab import files
    up = files.upload()
    !unzip -q -o ma-failure-repo.zip
    %cd ma-failure
!pip install -q -r requirements.txt
!ls

## Part A: results from the saved outputs (real-time design, as in the manuscript)

In [ ]:
import pandas as pd
from IPython.display import Image, display
pd.set_option("display.width", 200)
C = "outputs/corrected/"
print("Table 1 - out-of-sample performance, fiscal 2017-2022 (95% bootstrap intervals)")
display(pd.read_csv(C + "table1.csv", index_col=0).round(3))
print("AUC differences"); display(pd.read_csv(C + "auc_differences.csv").round(3))
print("AUC by year"); display(pd.read_csv(C + "auc_by_year.csv", index_col=0).round(3))
print("Settings chosen by real-time tuning"); display(pd.read_csv(C + "chosen_settings.csv").round(3))

In [ ]:
print("Table 2 - alert rule for the acquirer"); display(pd.read_csv(C + "table2_alert_rule.csv", index_col=0).T.round(3))
print("Case studies (fixed rules)"); display(pd.read_csv(C + "case_studies.csv").round(3))
print("Table 3 - stock-market tests (declared test as run, then the same specifications on the real-time predictions)")
display(pd.read_csv("outputs/stock_test/declared_test.csv").round(4))
display(pd.read_csv(C + "table3_stock.csv").round(4))
print("Robustness and pre-registered run"); display(pd.read_csv(C + "robustness.csv", index_col=0).round(3))
display(pd.read_csv("outputs/tuning/table1_scores.csv", index_col=0).round(3))
display(pd.read_csv("outputs/robustness/lag4_auc.csv", index_col=0).round(3))

In [ ]:
for f in ["fig1_auc_by_year", "fig2_failure_by_quintile", "fig3_drivers", "fig4_flag_rule", "fig5_bhar_by_tercile"]:
    display(Image(f"outputs/figures/{f}.png", width=750))

## Part B: full reproduction from raw data

Run the cells in order. Steps B1-B2 download about 1.4 GB from the SEC and take most of the time.

In [ ]:
# B1. SEC XBRL archive -> 10-K facts as first reported; firm-year panel and acquisition events
!python src/extract_sec.py
!python src/build_panel.py

In [ ]:
# B2. Industry codes and tickers (SEC), monthly prices (Yahoo) and market factor (Kenneth French)
!python src/fetch_meta.py
!python src/fetch_prices.py

In [ ]:
# B3. Optional: licensed Bloomberg prices of delisted acquirers (filled outputs/bloomberg_template.xlsx)
import os
os.makedirs("data/bloomberg", exist_ok=True)
try:
    from google.colab import files
    up = files.upload()                      # choose bloomberg_prices.xlsx, or cancel to skip
    for name in up:
        os.replace(name, "data/bloomberg/bloomberg_prices.xlsx")
except Exception as e:
    print("Skipped:", e)

In [ ]:
# B4. Models: development walk-forward (2014-2016) and locked final test (2017-2022)
!python src/run_all.py --period dev
!python src/run_all.py --period final --force

In [ ]:
# B5. Stock-market tests, pre-registered tuned run, real-time (corrected) walk-forward, tables, figures, checks
import os
if os.path.exists("data/bloomberg/bloomberg_prices.xlsx"):
    !python src/merge_bloomberg.py
!python src/stock_test.py --force
!python src/stock_test_median.py
!python src/tuning.py
!python src/losses_avoided.py
!python src/market_checks.py
!python src/robustness_lag.py
!python src/realtime.py
!python src/corrected_results.py
!python src/figures.py
!python tests/test_pipeline.py

Re-run Part A to display the reproduced tables and figures.